<a href="https://colab.research.google.com/github/GaNeShMs7-coder/BIS_1BM24CS100/blob/main/week4_BIS_1BM24CS100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import cv2
import numpy as np

# Load MRI image and reference mask in grayscale
img = cv2.imread("mri.png", 0)
ref_mask = cv2.imread("mask.png", 0)
ref = ref_mask > 127

N, T = 10, 30


# Fitness using Dice coefficient
def fitness(th):
    seg = img > th
    intersection = np.sum(seg & ref)
    return 2.0 * intersection / (np.sum(seg) + np.sum(ref) + 1e-9)


# Initialize wolves (threshold values between 0 and 255)
wolves = np.random.uniform(0, 255, N)

# Grey Wolf Optimization
for t in range(T):
    scores = np.array([fitness(w) for w in wolves])
    order = np.argsort(scores)[::-1]  # Higher Dice is better

    alpha = wolves[order[0]]
    beta = wolves[order[1]]
    delta = wolves[order[2]]

    a = 2.0 - 2.0 * (t / T)

    for i in range(N):
        positions = []
        for leader in [alpha, beta, delta]:
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2.0 * a * r1 - a
            C = 2.0 * r2
            d_leader = abs(C * leader - wolves[i])
            positions.append(leader - A * d_leader)

        wolves[i] = np.clip(np.mean(positions), 0, 255)

# Best solution
final_scores = np.array([fitness(w) for w in wolves])
best_idx = np.argmax(final_scores)
best = wolves[best_idx]

segmented = (img > best).astype(np.uint8) * 255
cv2.imwrite("segmented.png", segmented)

print("===== GWO MEDICAL IMAGE SEGMENTATION =====")
print("Optimal Threshold:", round(best, 2))
print("Dice Coefficient:", round(fitness(best), 4))
print("Maximum Iterations:", T)
print("Output: segmented.png")
print("Segmentation: SUCCESS")
print("==========================================")


===== GWO MEDICAL IMAGE SEGMENTATION =====
Optimal Threshold: 81.14
Dice Coefficient: 1.0
Maximum Iterations: 30
Output: segmented.png
Segmentation: SUCCESS
